##Quantizing an LLM in GGUF format.
#####GGUF is a format that allows to run LLM on CPU/GPU and more (compression teqniques i.e. Quantization)
    

In [ ]:
!!git clone https://github.com/ggerganov/llama.cpp

In [ ]:
# Use CUBLAS software that help to decide what machines are using
!cd llama.cpp && LLAMA_CUBLAS=1 make && pip install -r requirements.txt

In [ ]:
# huggingface snapshot
# download, and convert into 4-bit quantized model
# compressed teqniques - 1-bit for inference LLM (reduced computation power)

from huggingface_hub import snapshot_download
model_name = "Qwen/Qwen1.5-1.8B"

In [ ]:
methods = ["q4_k_m"]

In [ ]:
# define methods in list
# pass 4-bits and others, which will create diffrent files

base_model = "./original_model/"
quantized_path = "./quantized_model/"

In [ ]:
# downlaod the base model
snapshot_download(
    repo_id = model_name,
    local_dir = base_model,
    local_dir_use_symlinks = False
)

In [ ]:
original_model = quantized_path+'/FP16.gguf'

In [ ]:
# convert-hf-to-gguf.py
# block-wise quantization
!python llama.cpp/convert_hf_to_gguf.py ./original_model/ --outtype f16 --outfile ./quantized_model/FP16.gguf

In [ ]:
import os
# applying methods
quantized_path = "./quantized_model"
for m in methods:
  qtype = f"{quantized_path}/{m.upper()}.gguf"
  os.system("./llama.cpp/quantize "+quantized_path+"/FP16.gguf "+qtype+" "+m)

In [ ]:
# take the new quantized model, rag with chat-with-bob.txt
! ./llama.cpp/main -m ./quantized_model/Q4_K_M.gguf -n 90 --repeat_penalty 1.0 --color -i -r "User:" -f llama.cpp/prompts/chat-with-bob.txt

In [ ]:
!pip install huggingface-hub==0.34.3

In [ ]:
# Huggingface token
from huggingface_hub import notebook_login
notebook_login()

### Push quantized model to huggingface

In [ ]:
from huggingface_hub import HfApi, HfFolder, create_repo, upload_file

In [ ]:
# push model to huggingface with public access
model_path = "./quantized_model/FP16.gguf"
repo_name = "quantized-new-model3-qwen1.5-1.88B-GGUF"
repo_url = create_repo(repo_name, private=False)

In [ ]:
api = HfApi()

In [ ]:
# push our quantized model files to huggingface

api.upload_file(
    path_or_fileobj=model_path,
    path_in_repo="FP16.gguf",
    repo_id = "HFS26/quantized1-qwen1.5-1.88B-GGUF",
    repo_type="model"
)